# ADeLe lineages: generation on Kaggle

**Before running**
1. Settings: *Accelerator = GPU T4 x2*, *Internet = On* (needs a phone-verified account).
2. Add-ons -> Secrets: add `HF_TOKEN` and tick it for this notebook.
3. On Hugging Face, with the same account: accept the ADeLe battery terms and the Llama / Mistral licences.
4. Edit `REPO_URL` and `BATCH` below.

Interactive session: run the cells down to the smoke test.  
Full run: **Save Version -> Save & Run All (Commit)**. It runs without your browser and keeps `/kaggle/working` as the version's output.
Split the 10 models into 2-3 commits (by lineage) to stay under the 12-hour limit.

In [ ]:
REPO_URL = "https://github.com/<your-github-user>/adele-lineages.git"   # <- edit
BATCH = "llama,validation"     # e.g. "llama,validation" | "qwen" | "mistral" | "all"
RUN_SMOKE = True               # 5 items per format on one model first
BACKEND = "auto"               # auto | vllm | hf

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_HOME"] = "/kaggle/tmp/hf"          # big disk; /kaggle/working is capped at ~20 GB
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.makedirs("/kaggle/tmp/hf", exist_ok=True)

In [ ]:
%cd /kaggle/working
!if [ -d adele-lineages ]; then cd adele-lineages && git pull; else git clone {REPO_URL} adele-lineages; fi
%cd /kaggle/working/adele-lineages
!bash scripts/setup_kaggle.sh

In [ ]:
# Battery (gated) -> data/. data/subset_ids.csv comes from the repo (drawn in Codespaces).
!python src/data.py fetch
!test -f data/subset_ids.csv && wc -l data/subset_ids.csv || echo "MISSING data/subset_ids.csv - run data.py subset in Codespaces and push it" 

In [ ]:
# Resume: if you attached a previous version's output as an input dataset, copy its responses here.
import glob, shutil
os.makedirs("responses", exist_ok=True)
for f in glob.glob("/kaggle/input/*/adele-lineages/responses/*.jsonl"):
    dst = os.path.join("responses", os.path.basename(f))
    if not os.path.exists(dst):
        shutil.copy(f, dst); print("restored", dst)

In [ ]:
if RUN_SMOKE:
    !python src/generate.py --models llama-3.2-3b-instruct --limit 5 --backend {BACKEND} --keep-weights
    !head -c 600 responses/llama-3.2-3b-instruct.jsonl; echo
    !python src/score_mc.py --models llama-3.2-3b-instruct

In [ ]:
# Full batch. Each model runs in its own process; weights are deleted afterwards.
!python src/generate.py --models {BATCH} --backend {BACKEND}

In [ ]:
!ls -lh responses/
!for f in responses/*.jsonl; do echo "$f $(wc -l < $f)"; done
!zip -qr /kaggle/working/responses_{BATCH.replace(',', '_')}.zip responses
print("Download the zip from the Output tab (or attach this version as input to the next one).")